## 1. Infrastructure and reliability

This section sets up the connection to the AI model. It loads the API settings from a local `.env` file, keeps the key private, and shows how to handle errors and check token usage.

If any setting is missing, the notebook will not send a request.

In [1]:
# Uncomment and run only if these packages are missing from this kernel:
# %pip install openai python-dotenv

In [2]:
import os
from pathlib import Path

from dotenv import load_dotenv
from openai import APIError, AuthenticationError, OpenAI, RateLimitError

load_dotenv(Path.cwd() / ".env")
api_key = os.getenv("OPENAI_API_KEY", "").strip()
BASE_URL = os.getenv("OPENAI_BASE_URL", "").strip()
MODEL = os.getenv("MODEL_NAME", "").strip()
required_settings = {
    "OPENAI_API_KEY": api_key,
    "OPENAI_BASE_URL": BASE_URL,
    "MODEL_NAME": MODEL,
}
missing_settings = [name for name, value in required_settings.items() if not value]

if not missing_settings:
    client = OpenAI(
        api_key=api_key,
        base_url=BASE_URL,
        timeout=30.0,
        max_retries=0,
    )
    print("API configuration loaded from environment; credential is not displayed.")
else:
    client = None
    print(f"API configuration incomplete. Set these values in the local .env file: {', '.join(missing_settings)}. No API request was sent.")

API configuration loaded from environment; credential is not displayed.


The shared safe-call wrapper distinguishes authentication, rate-limit/quota, and other API errors. Failed calls return `None` and are reported, not presented as successful answers. When a call succeeds, the response helper prints token usage.
Requests use a 30-second timeout with no automatic retries, so an unavailable endpoint fails promptly.

In [3]:
def safe_call(messages, *, response_format=None):
    """Call the configured chat API and report expected failures clearly."""
    if client is None:
        print("Request not sent: configure an API key in the local .env file first.")
        return None

    options = {
        "model": MODEL,
        "messages": messages,
        "temperature": 0.2,
    }
    if response_format is not None:
        options["response_format"] = response_format

    try:
        return client.chat.completions.create(**options)
    except AuthenticationError as exc:
        print(
            "Authentication failed for the configured API provider. "
            "Confirm the key belongs to this provider, is active, and has API access. "
            f"Details: {exc}"
        )
    except RateLimitError as exc:
        print(f"Rate limit or quota reached. Check account limits and try again later. Details: {exc}")
    except APIError as exc:
        print(f"API request failed. Check provider settings and try again. Details: {exc}")
    return None


def print_answer(response):
    """Print a successful answer and its measured token usage."""
    if response is None:
        return
    print(response.choices[0].message.content)
    if response.usage is not None:
        print(
            f"Token usage — prompt: {response.usage.prompt_tokens}, "
            f"completion: {response.usage.completion_tokens}, "
            f"total: {response.usage.total_tokens}"
        )